# Итоговое решение

От сырых данных до `submission.csv`. Обоснование решений — в `EDA.ipynb`, `baseline.ipynb` и `experiments.ipynb`.

In [1]:
import re
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent / 'bot_detection_challenge'
sys.path.append(str(ROOT))
from metric import precision_at_recall

SEED = 42
DATE_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']

train = pd.read_csv(ROOT / 'data/train.csv', parse_dates=DATE_COLS)
test = pd.read_csv(ROOT / 'data/test.csv', parse_dates=DATE_COLS)
events_raw = pd.read_csv(ROOT / 'data/events.csv.gz', parse_dates=['event_ts'])
meta = pd.concat([train, test]).set_index('cookie_id')
y = train.target.to_numpy()

## Очистка событий

Подробнее в `EDA.ipynb`.
- Берём только события внутри окна куки: события после окна есть только в train, это утечка.
- Удаляем полные дубликаты (артефакт логирования) и `eid` (код `event_name`).
- Приводим `platform` к `web` / `android` / `ios` и сортируем события куки по времени.

In [2]:
start = events_raw.cookie_id.map(meta.window_start_ts)
end = events_raw.cookie_id.map(meta.window_end_ts)
events = events_raw[(events_raw.event_ts >= start) & (events_raw.event_ts < end)]

events = events.drop_duplicates().drop(columns='eid')

PLATFORM_MAP = {'android': 'android', 'ios': 'ios', 'iphone': 'ios', 'web': 'web', 'desktop': 'web'}
events['platform'] = events.platform.str.lower().map(PLATFORM_MAP)

events = events.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

## Признаки

Каждый признак — агрегат по событиям одной куки внутри её окна, поэтому train и test считаем вместе без утечки. Гипотезы и вклад групп — в `baseline.ipynb` и `experiments.ipynb`.

- `base` — сколько и каких событий, платформа, возраст куки;
- `timing` — паузы между событиями и ритм;
- `burst` — всплески активности;
- `search` — глубина листания, повторы запросов;
- `diversity` — разнообразие категорий и городов;
- `sequence` — переходы между типами событий, доля контактов;
- `pointer` — движение курсора на web;
- `ua` — тип клиента, ОС и версия из User-Agent.

Категориальные признаки (`platform`, `ua_client`, `ua_os`) передаём моделям как категории. Пропуски, например курсор у мобильных кук, оставляем NaN: бустинги обрабатывают их сами.

In [3]:
SESSION_GAP = 30 * 60


def base_features(events):
    counts = pd.crosstab(events.cookie_id, events.event_name)
    shares = counts.div(counts.sum(axis=1), axis=0)
    g = events.groupby('cookie_id')
    f = pd.DataFrame({'n_events': g.size(), 'platform': g.platform.first()})
    f['cookie_age_days'] = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400
    # у редких событий счётчик почти совпадает с долей, оставляем только долю
    counts = counts.drop(columns=['contact_chat_open', 'contact_message_sent', 'contact_phone_show', 'login'])
    return f.join(counts.add_prefix('cnt_')).join(shares.add_prefix('share_'))


def timing_features(events):
    dt = events.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    ev = events.assign(dt=dt, hour=events.event_ts.dt.hour)
    ev['dt_session'] = ev.dt.where(ev.dt < SESSION_GAP)
    ev['log_dt_session'] = np.log1p(ev.dt_session)
    g = ev.groupby('cookie_id')
    known = ev[ev.dt.notna()]
    n_sessions = 1 + (ev.dt >= SESSION_GAP).groupby(ev.cookie_id).sum()

    f = pd.DataFrame({
        'span_hours': (g.event_ts.max() - g.event_ts.min()).dt.total_seconds() / 3600,
        'dt_min': g.dt.min(),
        'dt_q10': g.dt.quantile(0.1),
        'dt_median': g.dt.median(),
        'dts_median': g.dt_session.median(),
        'dts_mean': g.dt_session.mean(),
        'dts_std': g.dt_session.std(),
        'log_dts_std': g.log_dt_session.std(),
        'dts_nunique_ratio': g.dt_session.nunique() / g.dt_session.count(),
        'share_dt_0': (known.dt == 0).groupby(known.cookie_id).mean(),
        'share_dt_le10': (known.dt <= 10).groupby(known.cookie_id).mean(),
        'n_active_hours': g.hour.nunique(),
        'share_night': (ev.hour < 6).groupby(ev.cookie_id).mean(),
    })
    f['dts_cv'] = f.dts_std / f.dts_mean
    f['events_per_session'] = g.size() / n_sessions
    f['events_per_active_hour'] = g.size() / f.n_active_hours
    return f


def burst_features(events):
    g = events.groupby('cookie_id')
    per_min = events.groupby(['cookie_id', events.event_ts.dt.floor('min')]).size()
    per_10min = events.groupby(['cookie_id', events.event_ts.dt.floor('10min')]).size()
    per_hour = events.groupby(['cookie_id', events.event_ts.dt.floor('h')]).size()
    n_active_minutes = per_min.groupby(level=0).size()
    return pd.DataFrame({
        'events_per_active_min': g.size() / n_active_minutes,
        'max_per_min': per_min.groupby(level=0).max(),
        'max_per_10min': per_10min.groupby(level=0).max(),
        'max_per_hour': per_hour.groupby(level=0).max(),
        'n_active_minutes': n_active_minutes,
    })


def search_features(events):
    s = events[events.event_name == 'search_results_view'].copy()
    g = s.groupby('cookie_id')
    s['same_query'] = s.search_query == g.search_query.shift()
    s['next_page'] = s.same_query & (s.search_page == g.search_page.shift() + 1)
    g = s.groupby('cookie_id')

    return pd.DataFrame({
        'page_max': g.search_page.max(),
        'page_mean': g.search_page.mean(),
        'share_page_gt1': (s.search_page > 1).groupby(s.cookie_id).mean(),
        'share_page_ge5': (s.search_page >= 5).groupby(s.cookie_id).mean(),
        'share_same_query': g.same_query.mean(),
        'share_next_page': g.next_page.mean(),
        'query_len_mean': s.search_query.str.len().groupby(s.cookie_id).mean(),
        'searches_per_query': g.size() / g.search_query.nunique(),
        'query_nunique_per_cat': g.search_query.nunique() / g.item_category.nunique().replace(0, np.nan),
    })


def entropy(events, col):
    counts = events.dropna(subset=[col]).groupby(['cookie_id', col]).size()
    p = counts / counts.groupby(level=0).transform('sum')
    return (-p * np.log(p)).groupby(level=0).sum()


def diversity_features(events):
    g = events.groupby('cookie_id')
    items = events[events.item_id.notna()]
    gi = items.groupby('cookie_id')
    category_counts = events.groupby(['cookie_id', 'item_category']).size()
    is_pro = items.seller_type.eq('pro').astype(float).where(items.seller_type.notna())

    return pd.DataFrame({
        'n_categories': g.item_category.nunique(),
        'n_locations': g.item_location.nunique(),
        'location_entropy': entropy(events, 'item_location'),
        'top_category_share': category_counts.groupby(level=0).max() / category_counts.groupby(level=0).sum(),
        'item_events_per_item': gi.size() / gi.item_id.nunique(),
        'share_pro_seller': is_pro.groupby(items.cookie_id).mean(),
        'n_seller_types': g.seller_type.nunique(),
    })


SHORT_NAMES = {'search_results_view': 'srch', 'item_view': 'item', 'photo_swipe': 'photo', 'seller_page_view': 'seller'}
TRANSITIONS = ['srch>item', 'item>srch', 'item>item', 'srch>srch', 'item>photo', 'item>seller']


def sequence_features(events):
    name = events.event_name.map(SHORT_NAMES).fillna(events.event_name)
    prev = name.groupby(events.cookie_id).shift()
    pair = prev + '>' + name
    n_transitions = pair.notna().groupby(events.cookie_id).sum()

    f = pd.crosstab(events.cookie_id, pair).reindex(columns=TRANSITIONS, fill_value=0).div(n_transitions, axis=0)
    f.columns = ['tr_' + c.replace('>', '_to_') for c in f.columns]
    f['share_repeat_event'] = (prev == name).groupby(events.cookie_id).sum() / n_transitions
    f['share_contact'] = events.event_name.str.startswith('contact').groupby(events.cookie_id).mean()
    f['items_per_search'] = ((events.event_name == 'item_view').groupby(events.cookie_id).sum()
                             / (events.event_name == 'search_results_view').groupby(events.cookie_id).sum())
    return f


def pointer_features(events):
    web = events[events.platform == 'web']
    p = web.dropna(subset=['pointer_x'])
    step = np.hypot(p.groupby('cookie_id').pointer_x.diff(), p.groupby('cookie_id').pointer_y.diff())
    g = p.assign(step=step).groupby('cookie_id')

    return pd.DataFrame({
        'pointer_share': web.pointer_x.notna().groupby(web.cookie_id).mean(),
        'px_mean': g.pointer_x.mean(),
        'px_std': g.pointer_x.std(),
        'px_max': g.pointer_x.max(),
        'py_mean': g.pointer_y.mean(),
        'py_std': g.pointer_y.std(),
        'pointer_step_mean': g.step.mean(),
    })


SCRIPT_UA = re.compile(r'python-requests|curl|node-fetch|urllib|Scrapy|Go-http-client', re.I)
BROWSERS = ['HeadlessChrome', 'YaBrowser', 'Edg', 'OPR', 'Firefox', 'Chrome', 'Safari']
OS_NAMES = {'Android': 'android', 'iPhone': 'ios', 'Windows': 'windows', 'Mac OS': 'mac', 'Linux': 'linux'}


def ua_client(ua):
    if ua.startswith('Avito/'):
        return 'avito_app'
    if SCRIPT_UA.search(ua):
        return 'script'
    return next((b for b in BROWSERS if b in ua), 'other')


def ua_os(ua):
    return next((name for marker, name in OS_NAMES.items() if marker in ua), 'other')


def ua_features(events):
    g = events.groupby('cookie_id').user_agent
    main_ua = g.agg(lambda s: s.value_counts().index[0])
    client = main_ua.map(ua_client)
    return pd.DataFrame({
        'ua_client': client,
        'ua_os': main_ua.map(ua_os),
        'ua_version': main_ua.str.extract(r'(?:Avito|Chrome|Firefox|Version)/(\d+)')[0].astype(float),
        'is_app': (client == 'avito_app').astype(int),
        'n_user_agents': g.nunique(),
        'has_script_ua': g.agg(lambda s: s.str.contains(SCRIPT_UA).any()).astype(int),
    })

In [4]:
features = pd.concat([
    base_features(events), timing_features(events), burst_features(events), search_features(events),
    diversity_features(events), sequence_features(events), pointer_features(events), ua_features(events),
], axis=1)

CAT_COLS = ['platform', 'ua_client', 'ua_os']
features[CAT_COLS] = features[CAT_COLS].fillna('missing')

X_train = features.reindex(train.cookie_id).reset_index(drop=True)
X_test = features.reindex(test.cookie_id).reset_index(drop=True)
print('признаков:', X_train.shape[1])

признаков: 76


Из пар дублей (корреляция Спирмена > 0.95) оставлен один признак.

## Модель

Ранговый блендинг CatBoost и LightGBM: модели ошибаются по-разному, и среднее рангов устойчивее каждой из них. Сид фиксирован (`SEED = 42`).

In [5]:
def make_catboost():
    return CatBoostClassifier(iterations=1600, learning_rate=0.015, depth=6, random_seed=SEED, verbose=0,
                              thread_count=-1, allow_writing_files=False, cat_features=CAT_COLS)


def make_lightgbm():
    return lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=30,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
                              random_state=SEED, verbose=-1, n_jobs=-1)


def to_lgb(X):
    X = X.copy()
    X[CAT_COLS] = X[CAT_COLS].astype('category')
    return X


def fit_predict(X_fit, y_fit, X_pred):
    """Обучает CatBoost и LightGBM и возвращает их вероятности для X_pred."""
    cat_pred = make_catboost().fit(X_fit, y_fit).predict_proba(X_pred)[:, 1]
    lgb_pred = make_lightgbm().fit(to_lgb(X_fit), y_fit).predict_proba(to_lgb(X_pred))[:, 1]
    return cat_pred, lgb_pred


def blend(cat_pred, lgb_pred):
    """Среднее рангов, отнормированное в [0, 1].

    Сумма рангов часто совпадает у разных кук, а метрика считает равные скоры одной группой.
    Поэтому добавляем вероятность CatBoost (< 1): она разбивает равные суммы и не меняет порядок остальных.
    """
    return (rankdata(cat_pred) + rankdata(lgb_pred) + cat_pred) / (2 * len(cat_pred) + 1)

## Валидация

Test идёт после train по времени, поэтому валидация тоже по времени: 4 фолда по 2 дня, обучение на всех предыдущих днях. Отдельно проверяем сплит с горизонтом в неделю, как у test. Метрику считаем по всем OOF-предсказаниям с 95% бутстрап-интервалом.

In [6]:
days = np.sort(train.window_start_ts.unique())
folds = []
for val_days in [days[6:8], days[8:10], days[10:12], days[12:14]]:
    folds.append((np.flatnonzero(train.window_start_ts < val_days[0]),
                  np.flatnonzero(train.window_start_ts.isin(val_days))))
testlike_fold = (np.flatnonzero(train.window_start_ts < '2026-04-13'),
                 np.flatnonzero(train.window_start_ts >= '2026-04-13'))


def evaluate(y_val, p_val):
    rng = np.random.default_rng(SEED)
    boot = [precision_at_recall(y_val[i], p_val[i]) for i in rng.integers(0, len(y_val), (1000, len(y_val)))]
    return {
        'p_at_r70': round(precision_at_recall(y_val, p_val), 3),
        'ci95': np.percentile(boot, [2.5, 97.5]).round(3),
        'pr_auc': round(average_precision_score(y_val, p_val), 3),
        'roc_auc': round(roc_auc_score(y_val, p_val), 3),
    }


oof_cat = np.full(len(y), np.nan)
oof_lgb = np.full(len(y), np.nan)
for tr, va in folds:
    oof_cat[va], oof_lgb[va] = fit_predict(X_train.iloc[tr], y[tr], X_train.iloc[va])

mask = ~np.isnan(oof_cat)
oof = np.full(len(y), np.nan)
oof[mask] = blend(oof_cat[mask], oof_lgb[mask])
print('P@R0.7 по фолдам:', [round(precision_at_recall(y[va], oof[va]), 3) for _, va in folds])

tr, va = testlike_fold
testlike_pred = blend(*fit_predict(X_train.iloc[tr], y[tr], X_train.iloc[va]))

pd.DataFrame({
    '4 фолда': evaluate(y[mask], oof[mask]),
    'горизонт как в test': evaluate(y[va], testlike_pred),
}).T

P@R0.7 по фолдам: [0.778, 0.77, 0.788, 0.758]


,p_at_r70,ci95,pr_auc,roc_auc
4 фолда,0.774,"[0.701, 0.839]",0.793,0.938
горизонт как в test,0.755,"[0.658, 0.831]",0.787,0.941


Итог: 0.77 [0.70, 0.84] на 4 фолдах и 0.76 на горизонте как у test.

Для сравнения, на тех же фолдах: quickstart — 0.09, CatBoost на базовых агрегатах — 0.23, CatBoost из `baseline.ipynb` — 0.77 [0.70, 0.81]. Основной прирост дали поведенческие признаки, блендинг делает решение устойчивее.

## Обучение на всём train и сабмит

In [7]:
score = blend(*fit_predict(X_train, y, X_test))
submission = pd.DataFrame({'cookie_id': test.cookie_id, 'score': score})

assert len(submission) == len(test) and submission.cookie_id.is_unique
assert set(submission.cookie_id) == set(test.cookie_id)
assert submission.score.between(0, 1).all() and submission.score.notna().all()

submission.to_csv(Path.cwd().parent / 'submission.csv', index=False)
submission.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.019248
1,ck_a76ee3b3e3e522fd,0.850909
2,ck_94c9a4d382689e82,0.831455
3,ck_8eaf9509ad9462a0,0.028211
4,ck_9a88a5a989cb5bc6,0.366840
